# Music4All-Onion — Kaggle V1, kết quả đầu tiên
Chỉ attach **userid_trackid_count.tsv.bz2** và **id_genres_tf-idf.tsv.bz2**. Không cần code ZIP, Git clone, Supabase, original MP3 hoặc lyrics. Bật Internet để cài dependencies; CPU là mặc định. Python 3.10–3.12.

V1 đọc tối đa 500.000 dòng đầu nguồn, lấy tối đa 1.000 user đầu tiên, tối đa 100 cặp/user; giữ user có >=5 cặp. Đây là sample theo thứ tự file, không ngẫu nhiên, không đại diện toàn dataset. Catalog genre vẫn giữ đầy đủ ID. Fit 32 factors × 8 iterations; eval tối đa 100 user/split. Không cam kết thời gian Kaggle trước khi chạy thật.

So sánh popularity, content, ALS, hybrid. Hash leave-two-out **không phải temporal**. Kết quả chỉ là baseline V1; metric có thể thấp hoặc bằng 0 và không được sửa số liệu. Không có tên bài/audio trong input.


In [ ]:
import sys, json, subprocess, os
from pathlib import Path
assert (3,10) <= sys.version_info[:2] < (3,13), 'Use Python 3.10–3.12'
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='4'
CODE=Path('/kaggle/working/music4all-v1-code')
CODE.mkdir(parents=True, exist_ok=True)
EMBEDDED = {'data_utils.py': '"""Local, streaming dataset utilities. No cloud credentials or environment files."""\n\nimport bz2\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy import sparse\n\nORIGINAL_TABLES = (\n    "id_information.csv",\n    "id_metadata.csv",\n    "id_genres.csv",\n    "id_tags.csv",\n    "id_lang.csv",\n    "listening_history.csv",\n)\n\n\ndef write_json(path, value):\n    Path(path).write_text(\n        json.dumps(value, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8"\n    )\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        while block := stream.read(8 * 1024 * 1024):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef read_catalog(root):\n    path = Path(root) / "id_information.csv"\n    with path.open(encoding="utf-8-sig", newline="") as stream:\n        sample = stream.read(16384)\n        stream.seek(0)\n        dialect = csv.Sniffer().sniff(sample, delimiters="\\t,;")\n        rows = csv.DictReader(stream, dialect=dialect)\n        fields = rows.fieldnames or []\n        aliases = {\n            "id": ("id", "track_id"),\n            "title": ("song", "name", "track_name", "title"),\n            "artist": ("artist", "artist_name"),\n            "album": ("album", "album_name"),\n        }\n        columns = {\n            key: next((name for name in names if name in fields), None)\n            for key, names in aliases.items()\n        }\n        if any(value is None for value in columns.values()):\n            raise ValueError(\n                "id_information.csv must contain ID, song/title, artist and album columns"\n            )\n        catalog = {}\n        for row in rows:\n            item = row[columns["id"]]\n            if not item or item != item.strip() or item in catalog:\n                raise ValueError("Empty, padded or duplicate original track ID")\n            if Path(item).name != item or item in {".", ".."} or "\\\\" in item:\n                raise ValueError("Unsafe original track ID")\n            catalog[item] = {\n                key: row[column] for key, column in columns.items() if key != "id"\n            }\n    return catalog\n\n\ndef read_onion_catalog(feature_path):\n    """Use every genre-file ID without inventing titles or playable audio."""\n    catalog = {}\n    with bz2.open(feature_path, "rt", encoding="utf-8") as stream:\n        reader = csv.reader(stream, delimiter="\\t", quoting=csv.QUOTE_NONE)\n        header = next(reader)\n        if header[0] != "id" or len(header) < 2:\n            raise ValueError("Expected Onion genre header: id + genre columns")\n        for row in reader:\n            item = row[0]\n            if (\n                not item\n                or item != item.strip()\n                or item in catalog\n                or len(row) != len(header)\n            ):\n                raise ValueError("Empty, duplicate or malformed Onion track ID/row")\n            catalog[item] = {\n                "title": None,\n                "artist": None,\n                "album": None,\n                "display_name": f"Track {item}",\n                "metadata_available": False,\n            }\n    if not catalog:\n        raise ValueError("Empty Onion catalog")\n    return catalog\n\n\ndef inventory(root, expected=109269, full=True):\n    root = Path(root)\n    catalog = read_catalog(root)\n    missing_tables = [\n        name\n        for name in ORIGINAL_TABLES\n        if not (root / name).is_file() or (root / name).stat().st_size == 0\n    ]\n    ids = set(catalog)\n    result = {\n        "catalog_tracks": len(ids),\n        "expected_tracks": expected,\n        "missing_tables": missing_tables,\n    }\n    for folder, extension in (("audios", ".mp3"), ("lyrics", ".txt")):\n        files = list((root / folder).rglob(f"*{extension}"))\n        files = [p for p in files if p.stem in ids]\n        present = {p.stem for p in files if p.stat().st_size > 0}\n        result[folder] = {\n            "nonempty_matching_ids": len(present),\n            "missing_ids": len(ids - present),\n            "duplicate_ids": len(files) - len({p.stem for p in files}),\n            "bytes": sum(p.stat().st_size for p in files),\n        }\n    result["complete"] = (\n        len(ids) == expected\n        and not missing_tables\n        and all(\n            result[key]["missing_ids"] == 0 and result[key]["duplicate_ids"] == 0\n            for key in ("audios", "lyrics")\n        )\n    )\n    result["verification"] = (\n        "Nonempty file/ID inventory; does not decode MP3 or establish playback permissions"\n    )\n    if full and not result["complete"]:\n        raise ValueError(\n            "Full original Music4All inventory failed: " + json.dumps(result)\n        )\n    return catalog, result\n\n\ndef load_features(path, item_ids):\n    """Read an ID + numerical TSV without materializing a dense catalog matrix."""\n    lookup = {item: index for index, item in enumerate(item_ids)}\n    row_indices, columns, values = [], [], []\n    seen = set()\n    opener = bz2.open if str(path).endswith(".bz2") else open\n    with opener(path, "rt", encoding="utf-8") as stream:\n        reader = csv.reader(stream, delimiter="\\t", quoting=csv.QUOTE_NONE)\n        header = next(reader)\n        if header[0] != "id" or len(header) < 2 or len(set(header)) != len(header):\n            raise ValueError(\n                "Expected numerical feature header: id + unique feature columns"\n            )\n        for row in reader:\n            if row[0] not in lookup:\n                continue\n            if row[0] in seen or len(row) != len(header):\n                raise ValueError("Duplicate or malformed feature row")\n            seen.add(row[0])\n            for column, raw in enumerate(row[1:]):\n                value = float(raw)\n                if not math.isfinite(value):\n                    raise ValueError("Nonfinite feature value")\n                if value != 0:\n                    row_indices.append(lookup[row[0]])\n                    columns.append(column)\n                    values.append(value)\n    matrix = sparse.csr_matrix(\n        (np.asarray(values, dtype=np.float32), (row_indices, columns)),\n        shape=(len(item_ids), len(header) - 1),\n    )\n    return matrix, header[1:]\n\n\ndef normalize_rows(matrix):\n    norms = np.sqrt(matrix.multiply(matrix).sum(axis=1)).A1\n    return sparse.diags(1 / np.where(norms > 0, norms, 1)).dot(matrix).tocsr()\n\n\ndef export_sparse(db, query, shape, output):\n    """Bounded result streaming; disk-backed buffers avoid 50M Python tuples."""\n    count = db.execute(f"SELECT count(*) FROM ({query}) q").fetchone()[0]\n    if not count:\n        matrix = sparse.csr_matrix(shape, dtype=np.float32)\n        sparse.save_npz(output, matrix)\n        return matrix\n    paths = [\n        Path(str(output) + suffix)\n        for suffix in (".rows.tmp", ".cols.tmp", ".values.tmp")\n    ]\n    buffers = [\n        np.memmap(path, dtype=dtype, mode="w+", shape=(count,))\n        for path, dtype in zip(paths, (np.int32, np.int32, np.float32))\n    ]\n    cursor = db.execute(query)\n    position = 0\n    while rows := cursor.fetchmany(100000):\n        batch = np.asarray(rows)\n        end = position + len(batch)\n        for column, target in enumerate(buffers):\n            target[position:end] = batch[:, column]\n        position = end\n    matrix = sparse.coo_matrix(\n        (buffers[2], (buffers[0], buffers[1])), shape=shape\n    ).tocsr()\n    sparse.save_npz(output, matrix)\n    for buffer in buffers:\n        buffer.flush()\n    del buffers\n    for path in paths:\n        path.unlink()\n    return matrix\n', 'prepare.py': '"""Prepare the complete catalog and bounded-memory sparse splits for Kaggle."""\n\nimport argparse\nimport bz2\nimport csv\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport duckdb\nfrom scipy import sparse\n\nfrom data_utils import (\n    export_sparse,\n    inventory,\n    load_features,\n    normalize_rows,\n    read_onion_catalog,\n    sha256,\n    write_json,\n)\n\n\ndef decompress(source, target):\n    with bz2.open(source, "rb") as reader, target.open("wb") as writer:\n        shutil.copyfileobj(reader, writer, length=8 * 1024 * 1024)\n\n\ndef prepare(config):\n    onion = Path(config["onion_root"])\n    feature_path = onion / "id_genres_tf-idf.tsv.bz2"\n    catalog_source = config.get("catalog_source", "original")\n    if catalog_source == "original":\n        catalog, report = inventory(\n            config["original_root"],\n            config.get("expected_tracks", 109269),\n            full=config.get("require_full_original", True),\n        )\n    elif catalog_source == "onion":\n        if config.get("require_full_original", False):\n            raise ValueError(\n                "Onion-only training cannot require the original audio inventory"\n            )\n        catalog = read_onion_catalog(feature_path)\n        report = {\n            "complete": False,\n            "checked": False,\n            "reason": "Onion-only training; original metadata/audio/lyrics not required or inspected",\n        }\n    else:\n        raise ValueError("catalog_source must be original or onion")\n    output = Path(config["output_root"])\n    prepared = output / "prepared"\n    if prepared.exists():\n        raise ValueError("Prepared output already exists; select a new output_root")\n    prepared.mkdir(parents=True)\n    scratch = Path(config["scratch_root"])\n    scratch.mkdir(parents=True, exist_ok=True)\n    # Unique scratch directory; never remove user-owned existing directories.\n    import tempfile\n\n    temporary = Path(tempfile.mkdtemp(prefix="prepare-", dir=scratch))\n    item_ids = sorted(catalog)\n    write_json(prepared / "catalog.json", catalog)\n    write_json(prepared / "item_ids.json", item_ids)\n    write_json(prepared / "original_inventory.json", report)\n    genre, names = load_features(feature_path, item_ids)\n    if genre.data.size and (genre.data < 0).any():\n        raise ValueError("Genre weights must be nonnegative")\n    sparse.save_npz(prepared / "genre.npz", normalize_rows(genre))\n    write_json(prepared / "genre_names.json", names)\n    split = config.get("split", "temporal")\n    if split not in {"temporal", "hash"}:\n        raise ValueError("split must be temporal or hash")\n    source_name = (\n        "userid_trackid_timestamp.tsv.bz2"\n        if split == "temporal"\n        else "userid_trackid_count.tsv.bz2"\n    )\n    source = onion / source_name\n    if not source.is_file():\n        raise ValueError(\n            f"Required source missing: {source_name}; no silent split fallback"\n        )\n    raw = temporary / "source.tsv"\n    print(\n        "Decompressing interaction file into scratch (not notebook output)", flush=True\n    )\n    decompress(source, raw)\n    with raw.open(encoding="utf-8") as stream:\n        header = next(csv.reader(stream, delimiter="\\t"))\n    expected = ["user_id", "track_id", "timestamp" if split == "temporal" else "count"]\n    if header != expected:\n        raise ValueError(f"Unexpected interaction header; expected {expected}")\n    db = duckdb.connect(str(temporary / "prepare.duckdb"))\n    try:\n        db.execute("SET memory_limit = \'4GB\'")\n        db.execute("SET threads = ?", [config.get("threads", 4)])\n        db.execute("SET preserve_insertion_order = false")\n        db.execute("SET TimeZone = \'UTC\'")\n        db.execute(\n            "CREATE TABLE catalog (track_id VARCHAR PRIMARY KEY, item_index INTEGER)"\n        )\n        # Bulk-load ID mapping instead of issuing 109k individual Python inserts.\n        catalog_rows = temporary / "catalog.tsv"\n        with catalog_rows.open("w", encoding="utf-8", newline="") as stream:\n            writer = csv.writer(stream, delimiter="\\t")\n            writer.writerow(["track_id", "item_index"])\n            writer.writerows((item, i) for i, item in enumerate(item_ids))\n        db.execute(\n            "INSERT INTO catalog SELECT track_id, item_index FROM read_csv(?, "\n            "delim=\'\\t\', header=true, columns={\'track_id\':\'VARCHAR\',\'item_index\':\'INTEGER\'})",\n            [str(catalog_rows)],\n        )\n        db.execute(\n            "CREATE VIEW source AS SELECT user_id, track_id, "\n            + ("timestamp" if split == "temporal" else "count")\n            + " AS value FROM read_csv(\'"\n            + str(raw).replace("\'", "\'\'")\n            + "\', delim=\'\\t\', header=true, all_varchar=true, quote=\'\', escape=\'\')"\n        )\n        if split == "temporal":\n            # Accept Unix seconds or ISO timestamps. Numeric timestamps are seconds,\n            # not milliseconds; out-of-range years are rejected below.\n            db.execute("""CREATE TABLE events AS\n                SELECT s.user_id, c.item_index,\n                  coalesce(try_cast(s.value AS TIMESTAMPTZ),\n                    to_timestamp(try_cast(s.value AS DOUBLE))) AS occurred_at\n                FROM source s JOIN catalog c USING(track_id)""")\n            bad = db.execute("""SELECT count(*) FROM events WHERE\n                user_id IS NULL OR trim(user_id) = \'\' OR occurred_at IS NULL\n                OR year(occurred_at) NOT BETWEEN 1970 AND 2100""").fetchone()[0]\n            if bad:\n                raise ValueError(f"Invalid in-catalog event rows: {bad}")\n            q1, q2 = (\n                config.get("train_quantile", 0.8),\n                config.get("validation_quantile", 0.9),\n            )\n            if not 0 < q1 < q2 < 1:\n                raise ValueError("Require 0 < train_quantile < validation_quantile < 1")\n            cuts = db.execute(\n                "SELECT quantile_disc(occurred_at, ?), quantile_disc(occurred_at, ?) FROM events",\n                [q1, q2],\n            ).fetchone()\n            if cuts[0] is None or cuts[0] >= cuts[1]:\n                raise ValueError(\n                    "Not enough distinct event timestamps for temporal splits"\n                )\n            db.execute(\n                """CREATE TABLE pairs AS SELECT user_id, item_index,\n                count(*) FILTER (WHERE occurred_at < ?)::FLOAT AS train_plays,\n                count(*) FILTER (WHERE occurred_at >= ? AND occurred_at < ?)::FLOAT AS val_plays,\n                count(*) FILTER (WHERE occurred_at >= ?)::FLOAT AS test_plays\n                FROM events GROUP BY user_id, item_index""",\n                [cuts[0], cuts[0], cuts[1], cuts[1]],\n            )\n            split_detail = {\n                "kind": "global_temporal",\n                "validation_start": str(cuts[0]),\n                "test_start": str(cuts[1]),\n                "duplicates": "identical source events retained as listens",\n            }\n        else:\n            db.execute("""CREATE TABLE raw_counts AS SELECT s.user_id, c.item_index,\n                try_cast(s.value AS BIGINT) AS plays, s.value\n                FROM source s JOIN catalog c USING(track_id)""")\n            bad = db.execute("""SELECT count(*) FROM raw_counts WHERE\n                user_id IS NULL OR trim(user_id) = \'\' OR plays IS NULL OR plays <= 0\n                OR NOT regexp_full_match(value, \'[0-9]+\')""").fetchone()[0]\n            if bad:\n                raise ValueError(f"Invalid in-catalog count rows: {bad}")\n            db.execute(\n                """CREATE TABLE ordered AS SELECT user_id, item_index, sum(plays) AS plays,\n                row_number() OVER (PARTITION BY user_id\n                    ORDER BY md5(user_id || \':\' || item_index::VARCHAR || \':\' || ?), item_index) AS position,\n                count(*) OVER (PARTITION BY user_id) AS total\n                FROM raw_counts GROUP BY user_id, item_index""",\n                [str(config.get("seed", 42))],\n            )\n            db.execute("""CREATE TABLE pairs AS SELECT user_id, item_index,\n                CASE WHEN position > 2 OR total < 3 THEN plays ELSE 0 END::FLOAT train_plays,\n                CASE WHEN position = 2 AND total >= 3 THEN plays ELSE 0 END::FLOAT val_plays,\n                CASE WHEN position = 1 AND total >= 3 THEN plays ELSE 0 END::FLOAT test_plays\n                FROM ordered""")\n            split_detail = {\n                "kind": "hash_leave_two_out",\n                "warning": "Non-temporal experiment; cannot measure forecasting or recent trends",\n            }\n        db.execute("""CREATE TABLE users AS SELECT user_id,\n            (row_number() OVER (ORDER BY user_id) - 1)::INTEGER AS user_index\n            FROM pairs GROUP BY user_id HAVING sum(train_plays) > 0""")\n        max_users = config.get("max_users")\n        if max_users is not None:\n            if not isinstance(max_users, int) or max_users < 1:\n                raise ValueError("max_users must be null or a positive integer")\n            db.execute("DELETE FROM users WHERE user_index >= ?", [max_users])\n        user_ids = [\n            row[0]\n            for row in db.execute(\n                "SELECT user_id FROM users ORDER BY user_index"\n            ).fetchall()\n        ]\n        if not user_ids:\n            raise ValueError("No training users")\n        write_json(prepared / "user_ids.json", user_ids)\n        shape = (len(user_ids), len(item_ids))\n        nnz = {}\n        for name, column, condition in (\n            ("train", "train_plays", "train_plays > 0"),\n            ("validation", "val_plays", "val_plays > 0 AND train_plays = 0"),\n            (\n                "test",\n                "test_plays",\n                "test_plays > 0 AND train_plays = 0 AND val_plays = 0",\n            ),\n        ):\n            query = f"SELECT u.user_index, p.item_index, p.{column} FROM pairs p JOIN users u USING(user_id) WHERE {condition}"\n            matrix = export_sparse(db, query, shape, prepared / f"{name}.npz")\n            nnz[name] = matrix.nnz\n            del matrix\n        source_rows = db.execute("SELECT count(*) FROM source").fetchone()[0]\n        retained_rows = db.execute(\n            "SELECT count(*) FROM source s JOIN catalog c USING(track_id)"\n        ).fetchone()[0]\n        all_users = db.execute("SELECT count(DISTINCT user_id) FROM pairs").fetchone()[\n            0\n        ]\n        manifest = {\n            "schema_version": 1,\n            "catalog_tracks": len(item_ids),\n            "training_users": len(user_ids),\n            "source_rows": source_rows,\n            "in_catalog_rows": retained_rows,\n            "users_without_train_or_limited": all_users - len(user_ids),\n            "pairs": nnz,\n            "split": split_detail,\n            "catalog_source": catalog_source,\n            "catalog_selection": f"All {catalog_source} catalog IDs; no popularity-based item selection",\n            "evaluation_target": "Unseen user-item discovery; validation excludes train, test excludes train + validation",\n            "genre_coverage": int((genre.getnnz(axis=1) > 0).sum()),\n            "source_sha256": {\n                source.name: sha256(source),\n                feature_path.name: sha256(feature_path),\n                **(\n                    {\n                        "id_information.csv": sha256(\n                            Path(config["original_root"]) / "id_information.csv"\n                        )\n                    }\n                    if catalog_source == "original"\n                    else {}\n                ),\n            },\n            "full_original_verified": report["complete"],\n        }\n        write_json(prepared / "manifest.json", manifest)\n        write_json(output / "config.json", config)\n        print(\n            json.dumps(\n                {\n                    "tracks": len(item_ids),\n                    "users": len(user_ids),\n                    "pairs": nnz,\n                    "split": split_detail,\n                },\n                indent=2,\n            )\n        )\n    finally:\n        db.close()\n        shutil.rmtree(temporary)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", type=Path, required=True)\n    args = parser.parse_args()\n    prepare(json.loads(args.config.read_text()))\n', 'train.py': '"""Train CPU/GPU ALS and evaluate popularity/content/ALS/hybrid on held-out pairs."""\n\nimport argparse\nimport importlib.metadata\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy import sparse\nfrom threadpoolctl import threadpool_limits\n\nfrom data_utils import sha256, write_json\nfrom predict import rank, score_components\n\n\ndef evaluate(model, train, validation, test, genre, config):\n    popularity = np.asarray(train.sum(axis=0)).ravel()\n    variants = {\n        "popularity": {"als": 0, "content": 0, "popularity": 1},\n        "content": {"als": 0, "content": 1, "popularity": 0},\n        "als": {"als": 1, "content": 0, "popularity": 0},\n        "hybrid": config["hybrid_weights"],\n    }\n    results = {}\n    for name, truth in (("validation", validation), ("test", test)):\n        eligible = np.flatnonzero(truth.getnnz(axis=1) > 0)\n        # Fixed seed subset, not just first IDs; same users for each method.\n        users = np.random.default_rng(config["seed"]).permutation(eligible)[\n            : config["eval_users"]\n        ]\n        accum = {\n            variant: {"recall": 0, "ndcg": 0, "mrr": 0, "precision": 0, "items": set()}\n            for variant in variants\n        }\n        started = time.monotonic()\n        for user in users:\n            components, excluded = score_components(\n                model, train, genre, popularity, int(user)\n            )\n            if name == "test":\n                # Frozen train-only model; mask validation observations when testing.\n                excluded.update(validation[user].indices.tolist())\n            relevant = set(truth[user].indices.tolist())\n            for variant, weights in variants.items():\n                ids, _, _ = rank(components, excluded, weights, config["k"])\n                hit = [int(item in relevant) for item in ids]\n                metrics = accum[variant]\n                metrics["recall"] += sum(hit) / len(relevant)\n                metrics["precision"] += sum(hit) / config["k"]\n                dcg = sum(\n                    value / math.log2(position + 2)\n                    for position, value in enumerate(hit)\n                )\n                ideal = sum(\n                    1 / math.log2(position + 2)\n                    for position in range(min(config["k"], len(relevant)))\n                )\n                metrics["ndcg"] += dcg / ideal\n                metrics["mrr"] += next(\n                    (1 / (position + 1) for position, value in enumerate(hit) if value),\n                    0,\n                )\n                metrics["items"].update(ids.tolist())\n        results[name] = {\n            "eligible_users": len(eligible),\n            "evaluated_users": len(users),\n            "seconds": round(time.monotonic() - started, 3),\n            "methods": {\n                variant: {\n                    **{\n                        f"{metric}_at_{config[\'k\']}": value / len(users)\n                        if len(users)\n                        else None\n                        for metric, value in metrics.items()\n                        if metric != "items"\n                    },\n                    "catalog_coverage": len(metrics["items"]) / train.shape[1],\n                }\n                for variant, metrics in accum.items()\n            },\n        }\n    return results\n\n\ndef train(config):\n    from implicit.als import AlternatingLeastSquares\n\n    output = Path(config["output_root"])\n    if json.loads((output / "config.json").read_text()) != config:\n        raise ValueError("Training config differs from prepared data; create a new run")\n    model_dir = output / "model"\n    if model_dir.exists():\n        raise ValueError("Model output already exists; use a new output_root")\n    model_dir.mkdir(parents=True)\n    train_matrix = sparse.load_npz(output / "prepared/train.npz").astype(np.float32)\n    validation = sparse.load_npz(output / "prepared/validation.npz")\n    test = sparse.load_npz(output / "prepared/test.npz")\n    genre = sparse.load_npz(output / "prepared/genre.npz")\n    confidence = train_matrix.copy()\n    confidence.data = 1 + config["alpha"] * np.log1p(confidence.data)\n    model = AlternatingLeastSquares(\n        factors=config["factors"],\n        regularization=config["regularization"],\n        iterations=config["iterations"],\n        alpha=1.0,\n        use_gpu=config["use_gpu"],\n        num_threads=config["threads"],\n        random_state=config["seed"],\n    )\n    started = time.monotonic()\n    with threadpool_limits(limits=1, user_api="blas"):\n        model.fit(confidence)\n        fit_seconds = time.monotonic() - started\n        if config["use_gpu"]:\n            model = model.to_cpu()\n        if (\n            not np.isfinite(model.user_factors).all()\n            or not np.isfinite(model.item_factors).all()\n        ):\n            raise ValueError("Nonfinite ALS factors")\n        # Checkpoint before evaluation so a long evaluation does not lose the fit.\n        model.save(model_dir / "als.npz")\n        metrics = evaluate(model, train_matrix, validation, test, genre, config)\n    versions = {\n        name: importlib.metadata.version(name)\n        for name in ("numpy", "scipy", "duckdb", "implicit", "scikit-learn")\n    }\n    write_json(\n        output / "metrics.json", {"fit_seconds": fit_seconds, "evaluation": metrics}\n    )\n    manifest = {\n        "model": "implicit-als-genre-hybrid-v1",\n        "versions": versions,\n        "confidence": "1 + alpha * log1p(train_count), implicit model alpha=1",\n        "config": config,\n        "files": {\n            str(path.relative_to(output)): sha256(path)\n            for path in sorted(output.rglob("*"))\n            if path.is_file() and path.name != "artifact_manifest.json"\n        },\n        "scope": "Recommender, not a genre classifier; no audio binary in output; not wired into FastAPI yet",\n        "evaluation": "Frozen train model; no validation/test refit or test-based tuning; unseen pairs only",\n    }\n    write_json(output / "artifact_manifest.json", manifest)\n    print(json.dumps({"fit_seconds": fit_seconds, "metrics": metrics}, indent=2))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", type=Path, required=True)\n    args = parser.parse_args()\n    train(json.loads(args.config.read_text()))\n', 'predict.py': '"""Load Kaggle artifacts for offline recommendations, without a network service."""\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy import sparse\n\n\ndef score_components(\n    model,\n    train,\n    genre,\n    popularity,\n    user_index=None,\n    genres=(),\n    genre_names=(),\n    recent=(),\n    confidence_alpha=20.0,\n):\n    items = genre.shape[0]\n    signal = sparse.csr_matrix((1, items), dtype=np.float32)\n    excluded = set(recent)\n    if user_index is not None:\n        signal = train[user_index].copy()\n        excluded.update(signal.indices.tolist())\n        signal.data = np.log1p(signal.data)\n    if recent:\n        session = sparse.csr_matrix(\n            (\n                [3 / (i + 1) for i in range(len(recent))],\n                ([0] * len(recent), list(recent)),\n            ),\n            shape=(1, items),\n            dtype=np.float32,\n        )\n        signal = signal + session\n    profile = np.asarray((signal @ genre).toarray()).ravel()\n    lookup = {name: i for i, name in enumerate(genre_names)}\n    for name in set(genres):\n        if name not in lookup:\n            raise ValueError("Unknown genre")\n        profile[lookup[name]] += 3\n    norm = np.linalg.norm(profile)\n    content = (\n        np.asarray(genre @ (profile / norm)).ravel() if norm > 0 else np.zeros(items)\n    )\n    collaborative = np.zeros(items, dtype=np.float32)\n    # No signal => no inferred CF profile. Existing users use fitted factors;\n    # recent item selections trigger fold-in using the same training confidence scale.\n    if signal.nnz:\n        if recent or user_index is None:\n            confidence = signal.copy()\n            confidence.data = 1 + confidence_alpha * confidence.data\n            # Fitted model uses alpha=1; the artifact stores scale separately below.\n            user = model.recalculate_user(0, confidence)\n        else:\n            user = model.user_factors[user_index]\n        collaborative = model.item_factors @ user\n        collaborative = np.maximum(collaborative, 0)\n        collaborative /= max(float(collaborative.max()), 1e-12)\n        # Items with no train observations have no trustworthy collaborative signal.\n        collaborative[popularity <= 0] = 0\n    popular = np.log1p(popularity)\n    popular /= max(float(popular.max()), 1e-12)\n    return {"als": collaborative, "content": content, "popularity": popular}, excluded\n\n\ndef rank(components, excluded, weights, k):\n    active = {\n        name: float(weight) if np.any(components[name] > 0) else 0.0\n        for name, weight in weights.items()\n    }\n    total = sum(active.values())\n    if total <= 0:\n        active = {"als": 0.0, "content": 0.0, "popularity": 1.0}\n        total = 1\n    active = {name: value / total for name, value in active.items()}\n    scores = sum(components[name] * value for name, value in active.items())\n    valid = np.ones(len(scores), dtype=bool)\n    if excluded:\n        valid[list(excluded)] = False\n    indices = np.flatnonzero(valid)\n    # Partial top-k, followed by stable ordering. Boundary ties resolved by index.\n    if len(indices) > k:\n        threshold = np.partition(scores[indices], -k)[-k]\n        higher = indices[scores[indices] > threshold]\n        ties = indices[scores[indices] == threshold][: k - len(higher)]\n        indices = np.concatenate((higher, ties))\n    indices = indices[np.lexsort((indices, -scores[indices]))][:k]\n    return indices, scores, active\n\n\nclass Recommender:\n    def __init__(self, run):\n        from implicit.cpu.als import AlternatingLeastSquares\n\n        run = Path(run)\n        self.config = json.loads((run / "config.json").read_text())\n        self.users = json.loads((run / "prepared/user_ids.json").read_text())\n        self.items = json.loads((run / "prepared/item_ids.json").read_text())\n        self.catalog = json.loads((run / "prepared/catalog.json").read_text())\n        self.genres = json.loads((run / "prepared/genre_names.json").read_text())\n        self.train = sparse.load_npz(run / "prepared/train.npz")\n        self.genre = sparse.load_npz(run / "prepared/genre.npz")\n        self.popularity = np.asarray(self.train.sum(axis=0)).ravel()\n        self.model = AlternatingLeastSquares.load(run / "model/als.npz")\n        # Model fit receives explicit 1+alpha*log1p(count), so its alpha remains 1.\n        self.confidence_alpha = self.config["alpha"]\n\n    def recommend(self, user_id=None, genres=(), recent=(), k=10):\n        if not 1 <= k <= 100:\n            raise ValueError("k must be between 1 and 100")\n        user = None if user_id is None else self.users.index(str(user_id))\n        recent_indices = [self.items.index(item) for item in dict.fromkeys(recent)]\n        if len(recent_indices) > 5:\n            raise ValueError("At most five recent tracks")\n        components, excluded = score_components(\n            self.model,\n            self.train,\n            self.genre,\n            self.popularity,\n            user,\n            genres,\n            self.genres,\n            recent_indices,\n            self.confidence_alpha,\n        )\n        indices, scores, active = rank(\n            components, excluded, self.config["hybrid_weights"], k\n        )\n        return [\n            {\n                "track_id": self.items[i],\n                **self.catalog[self.items[i]],\n                "score": float(scores[i]),\n                "components": {\n                    name: float(values[i]) for name, values in components.items()\n                },\n                "weights": active,\n                "audio_relative_path": (\n                    f"audios/{self.items[i]}.mp3"\n                    if self.config.get("catalog_source", "original") == "original"\n                    else None\n                ),\n                "playback_permission_verified": False,\n            }\n            for i in indices\n        ]\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--run", type=Path, required=True)\n    parser.add_argument("--user-id")\n    parser.add_argument("--genre", action="append", default=[])\n    parser.add_argument("--recent", action="append", default=[])\n    parser.add_argument("--limit", type=int, default=10)\n    args = parser.parse_args()\n    print(\n        json.dumps(\n            Recommender(args.run).recommend(\n                args.user_id, args.genre, args.recent, args.limit\n            ),\n            ensure_ascii=False,\n            indent=2,\n        )\n    )\n', 'quick_v1.py': '"""Bounded first-result experiment; source-order sampling is explicit, not full training."""\n\nimport argparse\nimport bz2\nimport csv\nimport json\nfrom pathlib import Path\n\nfrom data_utils import sha256, write_json\nfrom prepare import prepare\nfrom train import train\n\n\ndef sample_counts(source, destination, max_rows, max_users, max_pairs):\n    histories = {}\n    scanned = 0\n    if min(max_rows, max_users, max_pairs) < 1:\n        raise ValueError("Sampling limits must be positive")\n    with bz2.open(source, "rt", encoding="utf-8") as stream:\n        reader = csv.reader(stream, delimiter="\\t", quoting=csv.QUOTE_NONE)\n        if next(reader) != ["user_id", "track_id", "count"]:\n            raise ValueError("Unexpected count header")\n        for row in reader:\n            scanned += 1\n            if len(row) != 3 or not row[0].strip() or not row[1].strip():\n                raise ValueError("Malformed source row in V1 prefix")\n            user, item, raw_count = row\n            if (\n                not raw_count.isascii()\n                or not raw_count.isdigit()\n                or int(raw_count) <= 0\n            ):\n                raise ValueError("Invalid count in V1 prefix")\n            if user in histories or len(histories) < max_users:\n                history = histories.setdefault(user, {})\n                if item in history or len(history) < max_pairs:\n                    history[item] = history.get(item, 0) + int(raw_count)\n            if scanned >= max_rows:\n                break\n    histories = {\n        user: history for user, history in histories.items() if len(history) >= 5\n    }\n    if not histories:\n        raise ValueError(\n            "No user with >=5 sampled pairs; increase prefix/user/pair limits"\n        )\n    with bz2.open(destination, "wt", encoding="utf-8") as stream:\n        writer = csv.writer(stream, delimiter="\\t", lineterminator="\\n")\n        writer.writerow(["user_id", "track_id", "count"])\n        for user, history in histories.items():\n            for item, count in history.items():\n                writer.writerow([user, item, count])\n    return {\n        "source_rows_scanned": scanned,\n        "sample_users": len(histories),\n        "sample_pairs": sum(len(history) for history in histories.values()),\n        "selection": "First source-order users in a bounded source prefix; first pairs per user; not random or representative",\n        "limits": {"rows": max_rows, "users": max_users, "pairs_per_user": max_pairs},\n        "scope": "V1 bounded experiment, non-temporal; not full dataset training or current trends",\n    }\n\n\ndef run_v1(onion, output, scratch, settings):\n    if output.exists() or scratch.exists():\n        raise ValueError("Use new output AND sample scratch directories for V1")\n    for name in ("userid_trackid_count.tsv.bz2", "id_genres_tf-idf.tsv.bz2"):\n        if not (onion / name).is_file():\n            raise ValueError(f"Attach required Onion file: {name}")\n    scratch.mkdir(parents=True)\n    sample = scratch / "onion-sample"\n    sample.mkdir()\n    report = sample_counts(\n        onion / "userid_trackid_count.tsv.bz2",\n        sample / "userid_trackid_count.tsv.bz2",\n        settings["max_source_rows"],\n        settings["sample_users"],\n        settings["max_pairs_per_user"],\n    )\n    (sample / "id_genres_tf-idf.tsv.bz2").symlink_to(\n        (onion / "id_genres_tf-idf.tsv.bz2").resolve()\n    )\n    config = {\n        **settings,\n        "onion_root": str(sample),\n        "output_root": str(output),\n        "scratch_root": str(scratch / "duckdb"),\n    }\n    # Save provenance before preparation/training; included in artifact hashes.\n    output.mkdir(parents=True)\n    write_json(output / "v1_sampling.json", report)\n    print(json.dumps(report, indent=2), flush=True)\n    prepare(config)\n    train(config)\n    results = json.loads((output / "metrics.json").read_text())\n    if any(\n        results["evaluation"][name]["evaluated_users"] == 0\n        for name in ("validation", "test")\n    ):\n        raise ValueError(\n            "V1 has no eligible evaluation users; do not report empty metrics"\n        )\n    # A small summary is convenient to send without sharing model/history files.\n    with (output / "metrics_summary.csv").open("w", newline="") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(\n            [\n                "split",\n                "method",\n                "evaluated_users",\n                "recall_at_10",\n                "ndcg_at_10",\n                "mrr_at_10",\n                "precision_at_10",\n                "catalog_coverage",\n            ]\n        )\n        for split, evaluation in results["evaluation"].items():\n            for method, metrics in evaluation["methods"].items():\n                writer.writerow(\n                    [\n                        split,\n                        method,\n                        evaluation["evaluated_users"],\n                        *[\n                            metrics.get(key)\n                            for key in (\n                                "recall_at_10",\n                                "ndcg_at_10",\n                                "mrr_at_10",\n                                "precision_at_10",\n                                "catalog_coverage",\n                            )\n                        ],\n                    ]\n                )\n    manifest_path = output / "artifact_manifest.json"\n    manifest = json.loads(manifest_path.read_text())\n    manifest["files"]["metrics_summary.csv"] = sha256(output / "metrics_summary.csv")\n    write_json(manifest_path, manifest)\n    return results\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--onion-root", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--scratch", type=Path, required=True)\n    parser.add_argument(\n        "--config", type=Path, default=Path(__file__).parent / "configs/v1.json"\n    )\n    args = parser.parse_args()\n    run_v1(\n        args.onion_root, args.output, args.scratch, json.loads(args.config.read_text())\n    )\n', 'requirements-kaggle.txt': '# Python 3.10–3.12; same runtime pins as pyproject.toml\nduckdb==1.4.1\nnumpy==1.26.4\nscipy==1.13.1\nimplicit==0.7.2\nscikit-learn==1.5.2\nthreadpoolctl==3.6.0\npytz==2025.2\n', 'configs/v1.json': '{\n  "catalog_source": "onion",\n  "original_root": null,\n  "require_full_original": false,\n  "split": "hash",\n  "max_source_rows": 500000,\n  "sample_users": 1000,\n  "max_pairs_per_user": 100,\n  "max_users": null,\n  "seed": 42,\n  "factors": 32,\n  "iterations": 8,\n  "regularization": 0.05,\n  "alpha": 20.0,\n  "threads": 4,\n  "use_gpu": false,\n  "eval_users": 100,\n  "k": 10,\n  "hybrid_weights": {"als": 0.5, "content": 0.4, "popularity": 0.1}\n}\n'}
for name, text in EMBEDDED.items():
    target=CODE/name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding='utf-8')
subprocess.run([sys.executable,'-m','pip','install','-r',str(CODE/'requirements-kaggle.txt')],check=True)
print('Code ready. Training runs in a fresh subprocess, not in the notebook kernel.')


In [ ]:
# Auto-find the two attached input files. If multiple copies exist, set paths manually.
from datetime import datetime, timezone
def unique_input(name):
    matches=list(Path('/kaggle/input').rglob(name))
    assert len(matches)==1, f'Attach exactly one {name}; found {len(matches)}'
    return matches[0]
COUNT=unique_input('userid_trackid_count.tsv.bz2')
GENRE=unique_input('id_genres_tf-idf.tsv.bz2')
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
RUN=Path('/kaggle/working')/f'music4all-v1-{RUN_ID}'
SCRATCH=Path('/tmp')/f'music4all-v1-{RUN_ID}'
INPUT=SCRATCH/'attached'
INPUT.mkdir(parents=True)
(INPUT/COUNT.name).symlink_to(COUNT)
(INPUT/GENRE.name).symlink_to(GENRE)
CONFIG=CODE/'configs/v1.json'
settings=json.loads(CONFIG.read_text())
# Keep defaults for your first run. Do not choose parameters by looking at test metrics.
print({key:settings[key] for key in ['max_source_rows','sample_users','max_pairs_per_user','factors','iterations','eval_users']})


In [ ]:
subprocess.run([
    sys.executable,str(CODE/'quick_v1.py'),
    '--onion-root',str(INPUT),'--output',str(RUN),
    '--scratch',str(SCRATCH/'processing'),'--config',str(CONFIG)
],check=True)
print((RUN/'metrics_summary.csv').read_text())


In [ ]:
# Display a clear table without requiring pandas or importing numpy in this kernel.
import csv, html
from IPython.display import HTML, display
rows=list(csv.reader((RUN/'metrics_summary.csv').open()))
table='<table>'+''.join('<tr>'+''.join(f'<td>{html.escape(value)}</td>' for value in row)+'</tr>' for row in rows)+'</table>'
display(HTML(table))
print('V1 source-order sample, hash split; NOT a full-dataset or temporal result.')


In [ ]:
users=json.loads((RUN/'prepared/user_ids.json').read_text())
result=subprocess.run([sys.executable,str(CODE/'predict.py'),'--run',str(RUN),'--user-id',users[0],'--limit','10'],check=True,capture_output=True,text=True)
recommendations=json.loads(result.stdout)
(RUN/'recommendations_example.json').write_text(json.dumps(recommendations,ensure_ascii=False,indent=2))
print(result.stdout)


In [ ]:
# Download one ZIP containing model, mappings, matrices, configuration and results.
import shutil
from IPython.display import FileLink
archive=shutil.make_archive(str(RUN), 'zip', root_dir=RUN.parent, base_dir=RUN.name)
print('Save Version → Save & Run All to retain outputs. Download before closing.')
display(FileLink(archive))
print('Send: metrics_summary.csv + config.json + v1_sampling.json + screenshot of the table.')
print('Send model ZIP privately to backend; do not commit raw model/history/dataset to GitHub.')


## Nếu lỗi / làm gì tiếp
- Không tìm thấy file: Add Input chứa đúng hai tên file .tsv.bz2 rồi Run All; không dùng file ZIP chưa giải nén.
- Lỗi pip/mạng: bật Internet. Nếu Python 3.13+, chọn môi trường 3.10–3.12 tương thích pins.
- Có kết quả V1 trước, sau đó mới tăng sample hoặc làm temporal với timestamp trong notebook chính.
- Backend tích hợp artifact ALS và mapping; frontend hiển thị Track ID khi thiếu tên, chưa bật Play khi không có audio.
- Không sửa test metric để đẹp; chọn cấu hình bằng validation, giữ test cho báo cáo cuối.
